# 🔬 Colon Cancer Detection — Explainable Vision Transformer (ViT)
**Binary Classification:** Colon Adenocarcinoma vs Colon Benign Tissue

---
### Pipeline (Paper-Ready):
1. GPU check & library install
2. Mount Drive & load dataset
3. Imports, config & data transforms
4. DataLoaders + sample grid
5. ViT-Base/16 model definition
6. Loss / Optimizer / Scheduler
7. Training loop
8. **Figure 2** — Training curves (paper)
9. **Figure 3** — Confusion matrix (paper)
10. AUC-ROC curve
11. **Figure 6** — XAI: Attention Rollout heatmaps (paper)
12. **Figure 7** — Radar chart comparison (paper)
13. Save model & download all figures
14. Single-image inference
---


## ✅ Step 0 — GPU Check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — enable T4 GPU!')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 📦 Step 1 — Install Libraries


In [ ]:
!pip install -q timm transformers scikit-learn matplotlib seaborn


## 📁 Step 2 — Mount Drive & Verify Dataset

Expected folder structure:
```
dataset/
├── colon_aca/       ← adenocarcinoma images
└── colon_n/         ← normal benign images
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/dataset'   # ← CHANGE if needed

for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))])
        print(f'📂 {cls}: {count:,} images')


## 🔧 Step 3 — Imports & Config


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)
from PIL import Image
import warnings, os
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────
DATA_DIR    = '/content/drive/MyDrive/dataset'
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 10
LR          = 2e-4
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42
FIGURES_DIR = 'paper_figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Config ready — device:', device)
print(f'   Figures will be saved to: {FIGURES_DIR}/')


## 🖼️ Step 4 — Data Transforms, Dataset & Split


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transforms)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print('Classes found:', class_names)
print('Total images :', len(full_dataset))

n_total = len(full_dataset)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)
val_set.dataset.transform  = val_test_transforms
test_set.dataset.transform = val_test_transforms
print(f'Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}')


## 🚀 Step 5 — DataLoaders


In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
print('✅ DataLoaders ready')

# Quick sample grid
def imshow_batch(loader, class_names, n=8):
    imgs, labels = next(iter(loader))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    imgs = (imgs[:n] * std + mean).clamp(0, 1)
    fig, axes = plt.subplots(1, n, figsize=(20, 3))
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i].permute(1,2,0).numpy())
        ax.set_title(class_names[labels[i]], fontsize=9)
        ax.axis('off')
    plt.suptitle('Sample Training Images', fontweight='bold')
    plt.tight_layout(); plt.show()

imshow_batch(train_loader, class_names)


## 🤖 Step 6 — Load Pretrained ViT-Base/16


In [ ]:
model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=num_classes
)
model = model.to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model head          : {model.head}')
print(f'Total parameters    : {total_p:,}')
print(f'Trainable parameters: {trainable_p:,}')


## ⚙️ Step 7 — Loss, Optimizer & Scheduler


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print('✅ Loss / Optimizer / Scheduler ready')


## 🏋️ Step 8 — Training Loop


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs      = model(imgs)
            loss         = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct      += (outputs.argmax(1) == labels).sum().item()
            total        += labels.size(0)
    return running_loss / total, correct / total

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc  = 0.0
best_model_path = 'best_colon_vit.pth'

print(f'Training for {NUM_EPOCHS} epochs on {device}...')
print('='*70)
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    saved = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), best_model_path)
        saved = '  ✅ best saved'
    print(f'Ep {epoch:02d}/{NUM_EPOCHS} | Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | '
          f'Val Loss:{vl_loss:.4f} Acc:{vl_acc:.4f}{saved}')
print('='*70)
print(f'Best Val Accuracy: {best_val_acc:.4f}')


## 📈 Step 9 — Figure 2: Training Curves (Paper Figure)
> Saves `paper_figures/fig2_colon_training_curves.png` — used in LaTeX as Figure 2.


In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Colon Cancer ViT — Training Curves', fontsize=13, fontweight='bold', y=1.02)

# Loss
ax1.plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=5, label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   'r-o', linewidth=2, markersize=5, label='Val Loss')
ax1.set_title('Loss per Epoch', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(frameon=True); ax1.grid(True, alpha=0.3)
ax1.set_xticks(list(epochs_range))

# Accuracy
ax2.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', linewidth=2, markersize=5, label='Train Acc')
ax2.plot(epochs_range, [a*100 for a in history['val_acc']],   'r-o', linewidth=2, markersize=5, label='Val Acc')
ax2.set_title('Accuracy per Epoch (%)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim([50, 102])
ax2.legend(frameon=True); ax2.grid(True, alpha=0.3)
ax2.set_xticks(list(epochs_range))

plt.tight_layout()
out_path = f'{FIGURES_DIR}/fig2_colon_training_curves.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')


## 🧪 Step 10 — Test Evaluation


In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(device)
        out    = model(imgs)
        probs  = torch.softmax(out, dim=1).cpu()
        preds  = probs.argmax(dim=1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
print(f'\n🎯 Test Accuracy: {test_acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=class_names))


## 📊 Step 11 — Figure 3: Confusion Matrix (Paper Figure)
> Saves `paper_figures/fig3_colon_confusion_matrix.png` — used in LaTeX as Figure 3.


In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.8, annot_kws={'size': 14, 'weight': 'bold'}, ax=ax)
ax.set_title('Confusion Matrix — Colon Cancer Test Set\n(n = {:,})'.format(len(all_labels)),
             fontweight='bold', fontsize=12, pad=12)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.tick_params(labelsize=10)

plt.tight_layout()
out_path = f'{FIGURES_DIR}/fig3_colon_confusion_matrix.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')


## 📉 Step 12 — AUC-ROC Curve (Supporting Figure)


In [ ]:
# Binary ROC: use probability of class 1 (colon_aca)
fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2,
        label=f'ROC curve (AUC = {roc_auc:.4f})')
ax.plot([0,1],[0,1], color='navy', lw=1.5, linestyle='--', label='Random classifier')
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Colon Cancer (ViT-B/16)', fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = f'{FIGURES_DIR}/colon_roc_curve.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'AUC-ROC: {roc_auc:.4f}')
print(f'✅ Saved: {out_path}')


## 🔍 Step 13 — Figure 6a: XAI Attention Rollout Heatmaps (Paper Figure)
> Saves `paper_figures/fig6a_colon_xai_attention_rollout.png` — used in LaTeX as part of Figure 6.


In [ ]:
def get_attention_rollout(model, img_tensor, device, head_fusion='mean', discard_ratio=0.9):
    """Improved Attention Rollout for ViT-B/16."""
    model.eval()
    attention_maps = []

    def hook_fn(module, inp, out):
        B, N, C = inp[0].shape
        qkv = module.qkv(inp[0]).reshape(
            B, N, 3, module.num_heads, C // module.num_heads).permute(2, 0, 3, 1, 4)
        q, k, _ = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * (C // module.num_heads) ** -0.5
        attn = attn.softmax(dim=-1).detach().cpu()
        attention_maps.append(attn)

    hooks = [block.attn.register_forward_hook(hook_fn) for block in model.blocks]
    with torch.no_grad():
        output     = model(img_tensor.unsqueeze(0).to(device))
        pred_class = output.argmax(1).item()
        confidence = torch.softmax(output, dim=1).max().item()
    for h in hooks:
        h.remove()

    rollout = torch.eye(attention_maps[0].shape[-1])
    for attn in attention_maps:
        if   head_fusion == 'mean': a = attn.mean(dim=1)[0]
        elif head_fusion == 'min' : a = attn.min(dim=1).values[0]
        else                      : a = attn.max(dim=1).values[0]
        flat = a.view(-1)
        thresh = torch.quantile(flat, discard_ratio)
        a[a < thresh] = 0
        a = a + torch.eye(a.shape[0])
        a /= a.sum(dim=-1, keepdim=True)
        rollout = a @ rollout

    mask = rollout[0, 1:]
    gs   = int(mask.shape[0] ** 0.5)
    mask = mask.reshape(gs, gs).numpy()
    mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
    return mask, pred_class, confidence

print('✅ get_attention_rollout defined')


In [ ]:
def plot_xai_attention_rollout(model, dataset, class_names, device,
                               num_samples=6, save_path=None):
    """
    Publication-quality XAI figure: Original | Attention Rollout overlay.
    Designed to match Figure 6 in the research paper.
    """
    mean_np = np.array([0.485, 0.456, 0.406])
    std_np  = np.array([0.229, 0.224, 0.225])

    np.random.seed(42)
    # Try to get balanced samples (3 ACA + 3 benign)
    indices_by_class = {i: [] for i in range(len(class_names))}
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        indices_by_class[label].append(idx)

    samples_per_class = num_samples // len(class_names)
    chosen = []
    for cls_idx in range(len(class_names)):
        pool = indices_by_class[cls_idx]
        chosen.extend(np.random.choice(pool, min(samples_per_class, len(pool)), replace=False))
    np.random.shuffle(chosen)
    indices = chosen[:num_samples]

    fig, axes = plt.subplots(num_samples, 3, figsize=(13, num_samples * 3.2))
    fig.suptitle('XAI — Attention Rollout Heatmaps (Colon Cancer ViT)',
                 fontsize=13, fontweight='bold', y=1.01)

    col_titles = ['Original Image', 'Attention Rollout Overlay', 'Attention Map (Jet)']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold', pad=8)

    for row, idx in enumerate(indices):
        img_tensor, true_label = dataset[idx]
        img_np = np.clip(img_tensor.permute(1,2,0).numpy() * std_np + mean_np, 0, 1)

        mask, pred, conf = get_attention_rollout(model, img_tensor, device)
        mask_up = np.array(
            Image.fromarray((mask * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0

        ok    = pred == true_label
        color = '#1a7a1a' if ok else '#cc0000'
        mark  = '✓' if ok else '✗'

        # Col 0: Original
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_ylabel(f'True: {class_names[true_label]}',
                                 fontsize=9, labelpad=4)
        axes[row, 0].tick_params(left=False, bottom=False,
                                  labelleft=False, labelbottom=False)
        for sp in axes[row, 0].spines.values(): sp.set_visible(False)

        # Col 1: Overlay
        axes[row, 1].imshow(img_np)
        axes[row, 1].imshow(mask_up, alpha=0.45, cmap='jet')
        axes[row, 1].set_title(f'{mark} Pred: {class_names[pred]} ({conf*100:.1f}%)',
                                fontsize=9, color=color)
        axes[row, 1].axis('off')

        # Col 2: Raw mask
        im = axes[row, 2].imshow(mask_up, cmap='jet', vmin=0, vmax=1)
        axes[row, 2].axis('off')

    # Shared colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='Attention intensity')

    plt.tight_layout(rect=[0, 0, 0.91, 1])
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f'✅ Saved: {save_path}')
    plt.show()

out_path = f'{FIGURES_DIR}/fig6a_colon_xai_attention_rollout.png'
plot_xai_attention_rollout(model, test_set, class_names, device,
                            num_samples=6, save_path=out_path)


## 📊 Step 14 — Figure 7: Radar Chart Comparison (Paper Figure)
> Saves `paper_figures/fig7_radar_comparison.png` — used in LaTeX as Figure 7.
> Compares all reviewed models across 5 dimensions.


In [ ]:
import matplotlib.patches as mpatches
from matplotlib.path import Path
import matplotlib.patheffects as pe

categories = ['Accuracy', 'Interpretability\nDepth', 'Clinical\nReadiness',
              'Data\nEfficiency', 'Compute\nEfficiency']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

# Scores (0–10 scale) for each system
# [Accuracy, Interpretability, Clinical Readiness, Data Efficiency, Compute Efficiency]
systems = {
    'Alahmadi 2024 (ViT-B/L)'  : [9.3, 7.0, 6.0, 6.5, 5.5],
    'Baroni 2024 (ViT)'        : [9.1, 2.0, 5.0, 6.0, 5.5],
    'Martín 2025 (Transformer)': [9.6, 2.0, 5.5, 7.0, 5.0],
    'Hosny 2025 (Hybrid)'      : [9.4, 7.5, 7.0, 6.5, 6.0],
    'Habchi 2024 (Multimodal)' : [9.7, 8.0, 8.5, 5.5, 5.0],
    'Ours — Colon (ViT+XAI)'  : [9.87, 9.0, 7.5, 8.5, 7.0],
    'Ours — Lung (ViT+XAI)'   : [9.73, 9.0, 7.5, 8.5, 7.0],
}
colors = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#000075','#42d4f4']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_facecolor('#f9f9f9')

for (name, vals), color in zip(systems.items(), colors):
    vals_plot = vals + vals[:1]
    lw  = 2.8 if 'Ours' in name else 1.4
    ls  = '-'  if 'Ours' in name else '--'
    alp = 0.18 if 'Ours' in name else 0.04
    ax.plot(angles, vals_plot, color=color, linewidth=lw, linestyle=ls, label=name)
    ax.fill(angles, vals_plot, color=color, alpha=alp)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2','4','6','8','10'], fontsize=7, color='grey')
ax.yaxis.set_tick_params(labelsize=7)
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
ax.spines['polar'].set_visible(False)
ax.set_title('Model Comparison — 5-Dimension Radar Chart',
             fontsize=12, fontweight='bold', pad=20)

legend = ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15),
                   fontsize=8.5, frameon=True, framealpha=0.9)

plt.tight_layout()
out_path = f'{FIGURES_DIR}/fig7_radar_comparison.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Saved: {out_path}')


## 💾 Step 15 — Save Model & Download All Paper Figures


In [ ]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'class_names'      : class_names,
    'best_val_accuracy': best_val_acc,
    'test_accuracy'    : test_acc,
    'config': {
        'img_size'   : IMG_SIZE,
        'batch_size' : BATCH_SIZE,
        'num_epochs' : NUM_EPOCHS,
        'lr'         : LR,
        'num_classes': num_classes,
        'task'       : 'colon_binary'
    }
}, 'colon_cancer_vit_final.pth')
print('✅ Model saved: colon_cancer_vit_final.pth')

from google.colab import files
all_outputs = ['colon_cancer_vit_final.pth'] + [
    os.path.join(FIGURES_DIR, f)
    for f in os.listdir(FIGURES_DIR)
]
print('\n📦 Files to download:')
for f in sorted(all_outputs):
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'   {f}  ({size:.1f} KB)')
        files.download(f)
print('\n✅ All files downloaded.')


## 🔮 Step 16 — Single Image Inference with XAI


In [ ]:
def predict_with_xai(model, image_path, class_names, device):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    pil_img    = Image.open(image_path).convert('RGB')
    img_tensor = transform(pil_img)

    mask, pred_class, confidence = get_attention_rollout(model, img_tensor, device)
    mask_up = np.array(
        Image.fromarray((mask * 255).astype(np.uint8)).resize((224,224), Image.BILINEAR)) / 255.0

    model.eval()
    with torch.no_grad():
        out   = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(out, dim=1)[0].cpu().numpy()

    print(f'\n🔬 Prediction: {class_names[pred_class]} ({confidence*100:.1f}%)')
    for i, name in enumerate(class_names):
        bar = '█' * int(probs[i] * 40)
        print(f'   {name:20s} {probs[i]*100:5.1f}%  {bar}')

    img_np = np.array(pil_img.resize((224,224))) / 255.0
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    ax1.imshow(img_np); ax1.set_title('Original', fontweight='bold'); ax1.axis('off')
    ax2.imshow(img_np); ax2.imshow(mask_up, alpha=0.45, cmap='jet')
    ax2.set_title(f'Attention Rollout: {class_names[pred_class]} ({confidence*100:.1f}%)',
                  fontweight='bold'); ax2.axis('off')
    plt.tight_layout(); plt.show()

# from google.colab import files
# uploaded = files.upload()
# img_path = list(uploaded.keys())[0]
# predict_with_xai(model, img_path, class_names, device)
print('✅ predict_with_xai() ready — uncomment and run to test a single image')
